In [ ]:
!pip install catboost
!pip install seaborn
!pip install shap
!pip install lime

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
from statistics import mean
import matplotlib.pyplot as plt
import warnings
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.utils import resample

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report,ConfusionMatrixDisplay, \
                            precision_score, recall_score, f1_score, roc_auc_score,roc_curve,confusion_matrix


from sklearn import metrics 
from sklearn.model_selection import  train_test_split, RepeatedStratifiedKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer

from sklearn.preprocessing import StandardScaler, MinMaxScaler,RobustScaler
from sklearn.compose import ColumnTransformer
import sklearn as sklearn
sklearn.metrics.confusion_matrix
from matplotlib import pyplot as plt
import shap
from sklearn.feature_selection import RFE, f_regression
from sklearn.linear_model import (LinearRegression, Ridge, Lasso)
from sklearn.preprocessing import MinMaxScaler
warnings.filterwarnings("ignore")
%matplotlib inline

In [ ]:
data = pd.read_csv('CKD dataset.csv')
data.head(5)

In [ ]:
data = data.drop('Sr. No', axis=1)

In [ ]:
data.shape

In [ ]:
data.isna()

In [ ]:
plt.figure(figsize = (10,6))
sns.countplot(data = data, x='Target', order = data['Target'].value_counts().index)
plt.xticks(rotation = 45)
plt.title('Target Distribution')
plt.show()

In [ ]:
data['Sex'].value_counts().plot(kind = 'pie', autopct = '%.2f')

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split


# Splitting X and y for all Experiments
X = data.drop('Target', axis=1)
y = data['Target']

# Dictionary which contains models for experiment
models = {
    "Random Forest": RandomForestClassifier(),
    "Decision Tree": DecisionTreeClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "Logistic Regression": LogisticRegression(),
    "K-Neighbors Classifier": KNeighborsClassifier(),
    "XGBClassifier": XGBClassifier(), 
    "CatBoosting Classifier": CatBoostClassifier(verbose=False),
    "AdaBoost Classifier": AdaBoostClassifier()
}

def evaluate_clf(true, predicted):
    '''
    This function takes in true values and predicted values
    Returns: Accuracy, F1-Score, Precision, Recall, Roc-auc Score
    '''
    acc = accuracy_score(true, predicted) # Calculate Accuracy
    f1 = f1_score(true, predicted, average='weighted') # Calculate F1-score
    precision = precision_score(true, predicted, average='weighted') # Calculate Precision
    recall = recall_score(true, predicted, average='weighted')  # Calculate Recall
    roc_auc = roc_auc_score(true, predicted, multi_class='ovr') # Calculate Roc-AUC (for multiclass)
    return acc, f1, precision, recall, roc_auc

# Create a function which can evaluate models and return a report 
def evaluate_models(X, y, models):
    '''
    This function takes in X and y and models dictionary as input
    It splits the data into Train Test split
    Iterates through the given model dictionary and evaluates the metrics
    Returns: Dataframe which contains report of all models metrics
    '''
    # Separate dataset into train and test
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.50, random_state=42)
    
    models_list = []
    accuracy_list = []
    f1_list = []
    precision_list = []
    recall_list = []
    roc_auc_list = []
    
    for i in range(len(list(models))):
        model = list(models.values())[i]
        model.fit(X_train, y_train) # Train model

        # Make predictions
        y_test_pred = model.predict(X_test)

        # Model performance
        model_test_accuracy, model_test_f1, model_test_precision, model_test_recall, model_test_rocauc_score = evaluate_clf(y_test, y_test_pred)
        
        # Append the metrics to corresponding lists
        models_list.append(list(models.keys())[i])
        accuracy_list.append(model_test_accuracy)
        f1_list.append(model_test_f1)
        precision_list.append(model_test_precision)
        recall_list.append(model_test_recall)
        roc_auc_list.append(model_test_rocauc_score)
        
        # Print classification report
        print('Classification report:')
        print(list(models.keys())[i])
        print('- Accuracy: {:.4f}'.format(model_test_accuracy))
        print('- F1 score: {:.4f}'.format(model_test_f1))
        print('- Precision: {:.4f}'.format(model_test_precision))
        print('- Recall: {:.4f}'.format(model_test_recall))
        print('- Roc Auc Score: {:.4f}'.format(model_test_rocauc_score))
        print('='*35)
        print('\n')
        
    # Create a DataFrame for the report
    report = pd.DataFrame({
        'Model Name': models_list,
        'Accuracy': accuracy_list,
        'F1 Score': f1_list,
        'Precision': precision_list,
        'Recall': recall_list,
        'Roc Auc Score': roc_auc_list
    }).sort_values(by=["Accuracy"], ascending=False)
        
    return report

# Run the evaluation
report = evaluate_models(X, y, models)


In [ ]:
# Splitting X and y for all Experiments
X = data.drop('Target', axis=1)
y = data['Target']

# Dictionary which contains models for experiment
models = {
    "Random Forest": RandomForestClassifier(),
    "Decision Tree": DecisionTreeClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "Logistic Regression": LogisticRegression(),
    "K-Neighbors Classifier": KNeighborsClassifier(),
    "XGBClassifier": XGBClassifier(), 
    "CatBoosting Classifier": CatBoostClassifier(verbose=False),
    "AdaBoost Classifier": AdaBoostClassifier()
}


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

def evaluate_models_cv(X, y, models, cv=40):
    '''
    This function takes in X, y, and models dictionary as input.
    It performs cross-validation and evaluates the metrics.
    Returns: DataFrame which contains the report of all models' metrics.
    '''
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    
    results = []
    
    for model_name, model in models.items():
        acc_scores = []
        f1_scores = []
        precision_scores = []
        recall_scores = []
        roc_auc_scores = []
        
        for train_index, test_index in skf.split(X, y):
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train, y_test = y.iloc[train_index], y.iloc[test_index]
            
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            
            acc_scores.append(accuracy_score(y_test, y_pred))
            f1_scores.append(f1_score(y_test, y_pred, average='weighted'))
            precision_scores.append(precision_score(y_test, y_pred, average='weighted'))
            recall_scores.append(recall_score(y_test, y_pred, average='weighted'))
            
            if hasattr(model, "predict_proba"):
                y_prob = model.predict_proba(X_test)[:, 1]  # For binary case
                roc_auc_scores.append(roc_auc_score(y_test, y_prob))
            else:
                roc_auc_scores.append(roc_auc_score(y_test, y_pred, multi_class='ovr'))
        
        results.append({
            "Model Name": model_name,
            "Accuracy": sum(acc_scores) / len(acc_scores),
            "F1 Score": sum(f1_scores) / len(f1_scores),
            "Precision": sum(precision_scores) / len(precision_scores),
            "Recall": sum(recall_scores) / len(recall_scores),
            "Roc Auc Score": sum(roc_auc_scores) / len(roc_auc_scores)
        })
    
    report = pd.DataFrame(results).sort_values(by=["Accuracy"], ascending=False)
    return report

# Evaluate models with cross-validation
report = evaluate_models_cv(X, y, models)


In [ ]:
report

In [ ]:
final_model = RandomForestClassifier()
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.50,random_state=42)
final_model = final_model.fit(X_train, y_train)

In [ ]:
y_pred = final_model.predict(X_test)
y_pred_prob = final_model.predict_proba(X_test)[:,1]
print(classification_report(y_test, y_pred))


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import confusion_matrix

y_pred = final_model.predict(X_test)

# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=final_model.classes_)
disp.plot(cmap='Blues', values_format='d')

# Set title and labels
plt.title('Confusion Matrix')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')

# Adjust layout and save the plot
plt.tight_layout()
output_file_path = 'confusion_matrix_plot.png'
# plt.savefig(output_file_path, dpi=600)
plt.show()

In [ ]:
import shap

In [ ]:
# Convert X_test to a pandas DataFrame
X_test_df = pd.DataFrame(X_test, columns=X.columns)

In [ ]:
# Convert X_test to a pandas DataFrame
X_train_df = pd.DataFrame(X_train, columns=X.columns)

In [ ]:
# Define the SHAP explainer
explainer = shap.TreeExplainer(final_model)

In [ ]:
# Calculate SHAP values for the test data
shap_values = explainer.shap_values(X_test_df)

In [ ]:
# Summary Plot

In [ ]:
# Create the SHAP summary plot for positive class predictions
shap.summary_plot(shap_values[1], X_test_df)

In [ ]:
# Create the SHAP summary plot for negative class predictions
shap.summary_plot(shap_values[0], X_test_df)

In [ ]:
# Force Plot

In [ ]:
# Select the data point for which you want to create the force plot
data_point_index = 0

# Create the force plot for the selected data point
shap.initjs()  # Required to enable JavaScript for the plot

# Use the force_plot function to create the force plot for positive class predictions
force_plot = shap.force_plot(base_value=explainer.expected_value[1], shap_values=shap_values[1][data_point_index], features=X_test_df.iloc[data_point_index])

# Display the force plot
display(force_plot)

In [ ]:
# Select the data point for which you want to create the force plot
data_point_index = 0

# Create the force plot for the selected data point
shap.initjs()  # Required to enable JavaScript for the plot

# Use the force_plot function to create the force plot for negative class predictions
force_plot = shap.force_plot(base_value=explainer.expected_value[1], shap_values=shap_values[0][data_point_index], features=X_test_df.iloc[data_point_index])

# Display the force plot
display(force_plot)

In [ ]:
# Decision Plot

In [ ]:
data_point_index = 0  # You can change this index to visualize SHAP values for different data points

# Create a decision plot
shap.decision_plot(base_value=explainer.expected_value[1], 
                   shap_values=shap_values[1][data_point_index], 
                   features=X_test_df.iloc[data_point_index], 
                   link="logit")

In [ ]:
data_point_index = 0  # You can change this index to visualize SHAP values for different data points

# Create a decision plot
shap.decision_plot(base_value=explainer.expected_value[1], 
                   shap_values=shap_values[0][data_point_index], 
                   features=X_test_df.iloc[data_point_index], 
                   link="logit")

In [ ]:
# Waterfall plot

In [ ]:
# Choose a specific data point index (e.g., index 0)
data_point_index = 0

# Create a waterfall plot for the chosen data point
shap.waterfall_plot(shap.Explanation(values=shap_values[1][data_point_index],
                                     base_values=explainer.expected_value[1],
                                     data=X_test_df.iloc[data_point_index]),
                    max_display=20)  # You can adjust the 'max_display' parameter to control the number of features displayed

In [ ]:
# Choose a specific data point index (e.g., index 0)
data_point_index = 0

# Create a waterfall plot for the chosen data point
shap.waterfall_plot(shap.Explanation(values=shap_values[0][data_point_index],
                                     base_values=explainer.expected_value[0],
                                     data=X_test_df.iloc[data_point_index]),
                    max_display=20)  # You can adjust the 'max_display' parameter to control the number of features displayed

In [ ]:
# Contribution of influence of all features in RF model
fig = shap.summary_plot(shap_values, X_test_df, plot_type="bar", show=False)

In [ ]:
import lime
from lime import lime_tabular
from tabulate import tabulate

In [ ]:
# Initialize LIME explainer
explainer = lime_tabular.LimeTabularExplainer(X_test_df.values,
                                              feature_names=X_test_df.columns,
                                              class_names=['No CKD', 'CKD'],
                                              discretize_continuous=True)

# Choose a specific instance for explanation (e.g., the first instance in the test set)
instance = X_test_df.iloc[1]

# Get the model's prediction for the chosen instance
prediction = final_model.predict_proba(instance.values.reshape(1, -1))

# Explain the model's prediction using LIME
explanation = explainer.explain_instance(instance.values, final_model.predict_proba, num_features=len(X_test_df.columns))

In [ ]:
# Print the prediction and the explanation
print("Prediction:", prediction)
explanation.show_in_notebook()

In [ ]:
# Get the top features and their corresponding weights from the LIME explanation
top_features = explanation.as_list()

# Convert the list of tuples into a Pandas DataFrame
table_data = pd.DataFrame(top_features, columns=['Feature', 'Weight'])

# Sort the DataFrame by the absolute value of the weights to show the most important features first
table_data = table_data.iloc[table_data['Weight'].abs().argsort()[::-1]]

# Display the top 20 features in a presentable format
# top_20_features = table_data.head(20)
print(tabulate(table_data, headers='keys', tablefmt='fancy_grid', showindex=False))

In [ ]:
# # Get the top features and their corresponding weights from the LIME explanation
# top_features = explanation.as_list()

# # Convert the list of tuples into a Pandas DataFrame
# table_data = pd.DataFrame(top_features, columns=['Feature', 'Weight'])

# # Sort the DataFrame by the absolute value of the weights to show the most important features first
# table_data = table_data.iloc[table_data['Weight'].abs().argsort()[::-1]]

# # Select the top 20 features for the plot
# top_20_features = table_data.head(20)

# Create a bar plot to visualize the top 20 features and their weights
plt.figure(figsize=(12, 8))
plt.bar(table_data['Feature'], table_data['Weight'], color='skyblue', edgecolor='black')
plt.xticks(rotation=90)
plt.xlabel('Feature')
plt.ylabel('Weight')
plt.title('Features and their Weights')
plt.tight_layout()
plt.show()

In [ ]:
print('Job Done!')